# Domesticate CDS / multiple CDS

This notebook downloads the domestication script from GitHub, then removes restriction sites and user-defined avoiding motifs from existing CDS sequences by silent mutation. It contains both a single-CDS workflow and a CSV-based multiple-CDS workflow. It does not use the CodonTransformer codon optimization model. Kazusa codon usage is fetched with `python-codon-tables`.


In [ ]:
#@title Install dependencies
%%capture
!pip install -q biopython python-codon-tables


In [ ]:
#@title Download domestication source from GitHub
SOURCE_URL = "https://raw.githubusercontent.com/YuSugihara/CodonDomesticate/main/domesticate_cds.py"

import urllib.request
from pathlib import Path

source_path = Path("domesticate_cds.py")
urllib.request.urlretrieve(SOURCE_URL, source_path)
print("Downloaded:", SOURCE_URL)
print("Saved as:", source_path)


## Domesticate CDS

Use this section to domesticate one CDS entered directly in the notebook.


In [ ]:
#@title Configure single CDS domestication
cds = "ATGGGTCTCTAA" #@param {type:"string"}
organism = "Nicotiana benthamiana" #@param {type:"string"}
taxonomy_id = "" #@param {type:"string"}
enzyme_names = "BsaI;BpiI;Esp3I" #@param {type:"string"}
avoiding_motifs = "" #@param {type:"string"}
use_codon_usage = True #@param {type:"boolean"}
min_codon_freq = 0.2 #@param {type:"number"}
aa_change = "" #@param {type:"string"}
single_output_path = "single_domesticated.csv" #@param {type:"string"}


In [ ]:
#@title Run single CDS domestication
import csv
from google.colab import files

from domesticate_cds import (
    build_forbidden_motifs,
    domesticate_or_mutate_cds,
    download_kazusa_codon_usage,
    format_hits,
    format_mutations,
    get_avoiding_motifs_from_text,
    get_restriction_sites_from_enzyme_names,
    print_report,
    scan_forbidden_motifs,
)

restriction_sites = get_restriction_sites_from_enzyme_names(enzyme_names)
avoiding_sites = get_avoiding_motifs_from_text(avoiding_motifs)
forbidden_sites = build_forbidden_motifs(restriction_sites, avoiding_sites)

kazusa_codon_freqs = None
if use_codon_usage:
    taxid = int(taxonomy_id) if str(taxonomy_id).strip() else None
    kazusa_codon_freqs = download_kazusa_codon_usage(organism=organism, taxonomy_id=taxid)

final_cds, info = domesticate_or_mutate_cds(
    cds=cds,
    forbidden_sites=forbidden_sites,
    kazusa_codon_freqs=kazusa_codon_freqs,
    min_codon_freq=min_codon_freq,
    aa_change=aa_change,
)

print_report(info)

row = {
    "input_cds": info["input_cds"],
    "domesticated_cds": info["domesticated_cds"],
    "final_cds": final_cds,
    "protein_before": info["protein_before"],
    "protein_after": info["protein_after"],
    "num_sites_before": len(info["domestication_report"]["site_hits"]),
    "site_hits_before": format_hits(info["domestication_report"]["site_hits"]),
    "num_sites_after": len(scan_forbidden_motifs(final_cds, forbidden_sites)),
    "num_mutations": len(info["domestication_report"]["mutations"]),
    "mutations": format_mutations(info["domestication_report"]["mutations"]),
    "aa_change": info["aa_change"] or "",
}

with open(single_output_path, "w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(row))
    writer.writeheader()
    writer.writerow(row)

files.download(single_output_path)
print("Wrote:", single_output_path)


## Domesticate Multiple CDS

Use this section to upload a CSV file and domesticate multiple CDS records in one run. The `sequence` column is required; `name` and `aa_change` are optional.


In [ ]:
#@title Download batch CSV template
import csv
from google.colab import files

template_path = "template_cds.csv"
with open(template_path, "w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["name", "sequence", "aa_change"])
    writer.writeheader()
    writer.writerow({"name": "example_1", "sequence": "ATGGGTCTCTAA", "aa_change": ""})

files.download(template_path)
print("Template written:", template_path)


In [ ]:
#@title Upload input CSV
from google.colab import files

uploaded = files.upload()
dataset_path = next(iter(uploaded))
print("Uploaded:", dataset_path)


In [ ]:
#@title Configure batch domestication
batch_organism = "Nicotiana benthamiana" #@param {type:"string"}
batch_taxonomy_id = "" #@param {type:"string"}
batch_enzyme_names = "BsaI;BpiI;Esp3I" #@param {type:"string"}
batch_avoiding_motifs = "" #@param {type:"string"}
batch_use_codon_usage = True #@param {type:"boolean"}
batch_min_codon_freq = 0.2 #@param {type:"number"}
batch_output_path = "domesticated_cds_results.csv" #@param {type:"string"}


In [ ]:
#@title Run batch domestication
from google.colab import files
from domesticate_cds import main

args = [
    "batch",
    "--input-csv", dataset_path,
    "--output-csv", batch_output_path,
    "--organism", batch_organism,
    "--enzyme-names", batch_enzyme_names,
    "--avoiding-motifs", batch_avoiding_motifs,
    "--min-codon-freq", str(batch_min_codon_freq),
]
if str(batch_taxonomy_id).strip():
    args.extend(["--taxonomy-id", str(batch_taxonomy_id).strip()])
if not batch_use_codon_usage:
    args.append("--no-codon-usage")

exit_code = main(args)
if exit_code != 0:
    raise RuntimeError(f"Batch domestication failed with exit code {exit_code}")

files.download(batch_output_path)
print("Finished:", batch_output_path)
